# AnimationStudio - Phase 8: Image Generation & Visual Asset Pipeline

Turns a Phase 7 episode production plan into actual images.

- **Part A (mock, always runs):** generates a scene's per-shot images with
  the in-process `MockBackend`, then runs them through `ImageValidator`,
  `IdentityScorer`, and `ConsistencyManager`. Offline-green, no GPU.
- **Part B (real, GPU):** installs ComfyUI + fp8 Flux and generates images
  for a single scene via the ComfyUI backend, gated on the model download
  guards (N-08/N-10).

Two notes:
- Full Phase 1-3 libraries use `AnimationStudio_Colab.ipynb` /
  `AnimationStudio_Colab_Phase2.ipynb` — Phase 8 is the *episode-scene*
  image pass (a handful of shots), not the whole universe.
- A single T4 is ~2 min/image; the fp8 Flux model is ~17.25 GB. Both are
  accounted for in the cells below.

## Steps

1. Runtime -> Change runtime type -> T4 GPU (or better) for Part B.
2. In Cell 1 set `REPO_URL` to your GitHub clone URL.
3. Runtime -> Run all (Part A always runs; Part B runs on GPU).


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (colab-gpu is the only supported branch).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# colab-gpu -> fp8 Flux (16GB VRAM, best on T4).  master is deprecated/unused.
BRANCH = "colab-gpu"  #@param ["colab-gpu"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
UI_PORT = 8000  #@param {type:"integer"}

# The ONLY thing stored on Google Drive (free tier = 5 GB): the asset DB.
# Shortlisted/approved state survives session resets here.
DRIVE_ROOT = "/content/drive/MyDrive/AnimationStudio"  #@param {type:"string"}
DB = f"{DRIVE_ROOT}/catalog.db"

# The ~17.25 GB model cache. Free Drive cannot hold it -> keep on the Colab
# disk (re-downloaded after a VM reset). Set True only if you have space.
CACHE_MODELS_IN_DRIVE = False  #@param {type:"boolean"}

# ---- Part A (mock) episode to visualize ----
SEASON = 1  #@param {type:"integer"}
EPISODE_NUMBER = 1  #@param {type:"integer"}

# ---- Part B (real) scene selection ----
# Which scene index to render for real on the GPU.  Keep 0 on the free tier
# (a scene is several shots; each ~2 min/image on a T4).
REAL_SCENE_INDEX = 0  #@param {type:"integer"}
REAL_SHOTS = 3  #@param {type:"integer"}

# Part B needs the fp8 Flux model on disk.  Set True once you have:
#   a) a GPU runtime, and
#   b) the character/world libraries from Phases 1-3 (or run the lock
#      notebook first so identity scoring has references to compare).
RUN_REAL_GENERATION = False  #@param {type:"boolean"}

START_TUNNEL = True  #@param {type:"boolean"}

# Cell 9: push exported/approved output back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 to your GitHub repository before running.")


In [ ]:
#@title 2. Mount Google Drive (catalog.db only)

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive ready (holds catalog.db only):", DRIVE_ROOT)


In [ ]:
#@title 3. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

# Studio first (torch is already preinstalled on Colab), then light deps.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn", "timm", "diffusers", "transformers"])
print("Studio installed (branch:", BRANCH, ")")


In [ ]:
#@title 4. [PART A] Build the episode + per-shot prompts (mock)

# Reuses the Phase 7 planning chain entirely in-process, so Part A needs
# no GPU, no ComfyUI, and no network.

import sys
sys.path.insert(0, REPO)

from src.story_engine.generator import EpisodeGenerator
from src.production.blueprint_adapter import blueprint_to_episode
from src.production.pipeline import ProductionPipeline

gen = EpisodeGenerator(catalog_path=DB)
bp = gen.generate_episode(season=SEASON, episode_number=EPISODE_NUMBER)
ep = blueprint_to_episode(bp)
pipeline = ProductionPipeline()
prompts = pipeline.generate_prompts(ep)

print(f"episode {ep.id}: '{ep.title}' - {ep.scene_count} scenes, "
      f"{ep.shot_count} shots")
for shot_id, prompt in list(prompts.items())[:5]:
    print(f"  - {shot_id}: {prompt[:90]}...")
print()
print("Part A plan ready.")


In [ ]:
#@title 5. [PART A] Mock-generate + validate + score + enforce consistency

# Generates every shot with MockBackend, runs each image through the
# production ImageValidator, scores it with IdentityScorer, and enforces
# character/environment/style consistency.  Fully offline.

import io as _io
from PIL import Image as _PILImage

from src.generation_engine.base import GenerationInput
from src.generation_engine.mock_backend import MockBackend
from src.identity_engine.scorer import IdentityScorer
from src.image_generation.validator import ImageValidator
from src.image_generation.consistency import ConsistencyManager

backend = MockBackend()
validator = ImageValidator()
scorer = IdentityScorer(light=True)
consistency = ConsistencyManager()

results = []
for shot_id, prompt in prompts.items():
    out = backend.generate(GenerationInput(
        prompt=prompt,
        negative_prompt="blurry, distorted",
        seed=42,
        width=1024,
        height=1024,
        num_images=1,
    ))
    img = out.images[0]
    v = validator.validate_image(img)
    scores = scorer.score_all(img)
    brand = scorer.brand_score(img)
    results.append((shot_id, v.passed, brand.get("total", 0.0), scores))

print("=" * 72)
print("  MOCK GENERATION + VALIDATION + SCORING")
print("=" * 72)
passed = sum(1 for _sid, ok, _b, _s in results if ok)
print(f"  shots generated: {len(results)} | validation passed: {passed}/{len(results)}")
for shot_id, ok, brand, scores in results[:6]:
    print(f"    {shot_id:22s} valid={ok} brand={brand:.3f} plugins={len(scores)}")
print("=" * 72)
print("  ConsistencyManager state (mock has no locks unless Phases 1-3 ran)")
print("    locked characters:", consistency.locked_character_count())
print()
print("Part A complete - the visual pipeline code path ran end-to-end.")


In [ ]:
#@title 6. Install ComfyUI (Part B prerequisite)

if RUN_REAL_GENERATION:
    if not os.path.isdir(COMFY):
        run(["git", "clone", "--depth", "1",
             "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])
    print("ComfyUI ready at", COMFY)
else:
    print("RUN_REAL_GENERATION is off - skipping ComfyUI install (Part A only).")


In [ ]:
#@title 7. Download the fp8 Flux model (Part B, Colab disk NOT Drive)

if RUN_REAL_GENERATION:
    MODELS = {
        "colab-gpu": {
            "checkpoints/flux1-dev.safetensors":
                "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
        },
    }[BRANCH]

    import shutil

    cache_root = f"{DRIVE_ROOT}/models" if CACHE_MODELS_IN_DRIVE else f"{WORK}/models"
    for rel, url in MODELS.items():
        cached = f"{cache_root}/{rel}"
        link = f"{COMFY}/models/{rel}"
        if not (os.path.exists(cached) and os.path.getsize(cached) > 0):
            os.makedirs(os.path.dirname(cached), exist_ok=True)
            free_gb = shutil.disk_usage("/content").free / 1e9
            if free_gb < 4:
                raise SystemExit(
                    f"Only {free_gb:.1f} GB free on /content - cannot fit the "
                    "~17.25 GB fp8 model. Stop other runtimes or free disk space."
                )
            print(f"Downloading {rel} ...")
            run(["wget", "-q", "-c", "-O", cached, url])
        size_gb = os.path.getsize(cached) / 1e9
        if size_gb < 17.25 * 0.9:
            raise SystemExit(
                f"Model looks truncated: {size_gb:.2f} GB cached - re-run this "
                "cell (wget -c resumes) or delete the file and restart."
            )
        os.makedirs(os.path.dirname(link), exist_ok=True)
        if os.path.lexists(link) and not os.path.islink(link):
            os.remove(link)
        if not os.path.islink(link):
            try:
                os.symlink(cached, link)
            except OSError:
                shutil.copyfile(cached, link)
        print(f"OK {rel} ({size_gb:.2f} GB)")
    print("Model cache:", cache_root)
else:
    print("RUN_REAL_GENERATION is off - skipping fp8 Flux download.")


In [ ]:
#@title 8. Start ComfyUI server + verify GPU (Part B)

if RUN_REAL_GENERATION:
    import torch
    assert torch.cuda.is_available(), "GPU runtime required for Part B - set Runtime > Change runtime type > T4 GPU."
    print("GPU:", torch.cuda.get_device_name(0))

    import sys
    sys.path.insert(0, f"{REPO}/colab")
    from comfy_helpers import ensure_comfyui_up
    ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)
    print("ComfyUI ready on :" + str(COMFYUI_PORT))
else:
    print("RUN_REAL_GENERATION is off - skipping ComfyUI server + GPU check.")


In [ ]:
#@title 9. [PART B] Real image generation for one scene (ComfyUI)

if RUN_REAL_GENERATION:
    import sys
    sys.path.insert(0, f"{REPO}/colab")
    from comfy_helpers import ensure_comfyui_up
    ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

    from src.generation_engine.comfy_backend import ComfyUIBackend
    from src.generation_engine.base import GenerationInput
    from src.review_ui.combined_repo import check_comfyui

    if not check_comfyui(f"http://localhost:{COMFYUI_PORT}"):
        raise SystemExit("ComfyUI not reachable - start it in Cell 8 first.")

    backend = ComfyUIBackend(server_url=f"http://localhost:{COMFYUI_PORT}")

    scene = ep.scenes[REAL_SCENE_INDEX]
    shot_ids = [s.id for s in scene.shots][:REAL_SHOTS]
    print(f"Rendering scene {REAL_SCENE_INDEX} '{scene.title}' - "
          f"{len(shot_ids)} shot(s) on ComfyUI")

    os.chdir(REPO)
    for shot_id in shot_ids:
        prompt = prompts[shot_id]
        out = backend.generate(GenerationInput(
            prompt=prompt,
            negative_prompt="blurry, distorted, low quality",
            seed=1234,
            width=1024,
            height=1024,
            num_images=1,
        ), asset_type="scene")
        ok = bool(out.images)
        print(f"  {shot_id}: {'OK' if ok else 'EMPTY'} ({len(out.images)} image(s))")

    # Keep a running reference image dir for the Review UI to show.
    os.makedirs(f"{REPO}/Assets/scenes/{ep.id}", exist_ok=True)
    print()
    print("Part B done - exported scene images land in Assets/scenes/<episode>.")
else:
    print("RUN_REAL_GENERATION is off - skipping real generation (Part A only).")


In [ ]:
#@title 10. Launch the Review UI and tunnel (view approved assets)

from src.review_ui.app import create_app
from src.universe.batch_generator import resolve_backend

backend_kind = "comfyui" if RUN_REAL_GENERATION else "mock"
app = create_app(
    db_path=DB,
    generation_backend=resolve_backend(backend_kind,
                                       comfyui_url=f"http://localhost:{COMFYUI_PORT}"),
    universe_dir=f"{REPO}/Universe",
    world_dir=f"{REPO}/World",
    assets_dir=f"{REPO}/Assets",
    persist_generated_images=True,
)

import socket
import threading
import uvicorn

ui_alive = False
probe = socket.socket()
probe.settimeout(2)
try:
    probe.connect(("127.0.0.1", UI_PORT))
    ui_alive = True
except Exception:
    ui_alive = False
finally:
    probe.close()

if ui_alive:
    print(f"Review UI already running on :{UI_PORT}")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=UI_PORT, log_level="warning")
    threading.Thread(target=uvicorn.Server(config).run, daemon=True).start()
    print(f"Review UI starting on :{UI_PORT} ...")

if START_TUNNEL:
    import re
    import shutil

    if not shutil.which("lt"):
        run(["apt-get", "install", "-y", "-qq", "nodejs", "npm"])
        run(["npm", "install", "-g", "--silent", "localtunnel"])
    lt = shutil.which("lt") or "lt"

    def open_tunnel(port, name):
        out = open(f"{WORK}/{name}.log", "w")
        return subprocess.Popen([lt, "--port", str(port)],
                                stdout=out, stderr=subprocess.STDOUT)

    p1 = open_tunnel(COMFYUI_PORT, "tunnel_comfyui")
    p2 = open_tunnel(UI_PORT, "tunnel_ui")

    urls = {}
    for _ in range(30):
        time.sleep(2)
        for name in ("tunnel_comfyui", "tunnel_ui"):
            txt = open(f"{WORK}/{name}.log").read()
            found = re.findall(r"https://[a-z0-9-]+\.loca\.lt", txt)
            if name not in urls or not urls[name]:
                urls[name] = found
        if urls.get("tunnel_comfyui") and urls.get("tunnel_ui"):
            break

    for name in ("tunnel_comfyui", "tunnel_ui"):
        found = urls.get(name) or []
        print(name, "->", found if found else "no URL yet")
    print("Review UI:", urls.get("tunnel_ui"))
    print("ComfyUI:  ", urls.get("tunnel_comfyui"))


In [ ]:
#@title 11. Sync approved output (GitHub push or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import auto_sync

    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL,
              message=f"Phase 8 scene images {datetime.now():%Y-%m-%d %H:%M}")
else:
    import zipfile
    from google.colab import files

    zip_path = f"{WORK}/phase8_export.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{REPO}/catalog.db", "catalog.db")
        for root, _dirs, names in os.walk(f"{REPO}/Assets"):
            for name in names:
                full = os.path.join(root, name)
                z.write(full, os.path.relpath(full, REPO))
    files.download(zip_path)
    print("Downloaded phase8_export.zip (catalog.db + Assets).")


## Next steps

- **Part A always ran** - the mock visual pipeline (generate -> validate ->
  score -> consistency) proved the full Phase 8 code path without GPU.
- **Part B needs `RUN_REAL_GENERATION = True`** in Cell 1 plus a GPU runtime.
  Once Phases 1-3 libraries exist, Part B renders a real scene's shots via
  ComfyUI + fp8 Flux.
- **Approve assets in the Review UI** (Cell 10 tunnel) then re-run Cell 11.
- **Episode renders** build on this: see `AnimationStudio_Colab_Phase7.ipynb`
  for the planning chain, and the upcoming Phases 9-12 for animation/export.

## Troubleshooting

- Part B skipped silently: set `RUN_REAL_GENERATION = True` in Cell 1.
- CUDA out of memory: lower `REAL_SHOTS` in Cell 1, or switch to the L4/A100
  runtime.
- Model download stalls: re-run Cell 7 (`wget -c` resumes into the cache).
- Drive full: only `catalog.db` should be on Drive; delete any old `models/`
  cache under `DRIVE_ROOT` to reclaim ~17.25 GB.
